# 🌍 Week 5 — The Internet & APIs
**Carojon Academy · Python, AI & the Web**

> **Today's goal:** make your programs **talk to other computers** on the internet, starting with a **live weather app**. This is the exact skill you need to talk to AI in Week 6.

## 🗺️ You are here: Week 5 of 8
`1 ✅` → `2 ✅` → `3 ✅` → `4 ✅` → **`5 Internet & APIs`** → 6 AI → 7 HTML/CSS → 8 JavaScript + Capstone

## 🔁 Warm-up
1. What does `try / except` do?
2. How do you add a new key `"phone"` to a dictionary `d`?
3. Name the 3 parts of every app.
<details><summary>Answers</summary>Catches errors so the program doesn't crash · `d["phone"] = "0801..."` · input → function → output</details>

In [ ]:
from helpers import *
import gradio as gr
import random

## 1️⃣ Libraries — borrowing other people's code

**Analogy:** You don't bake your own bricks to build a house. You **buy bricks**. 🧱 A **library** (or **package**) is a box of ready-made code. Programmers worldwide share them.

Two kinds:
- **Standard library:** comes with Python (`random`, `math`, `datetime`)
- **Third-party packages:** installed with **`uv add`** (like `gradio`, `requests`)

In [ ]:
import math
import datetime

print(math.sqrt(144))
print(math.pi)
print(datetime.date.today())
print(datetime.datetime.now().strftime("%A, %d %B %Y"))

### 📦 Installing a package with `uv`
We need a package called **`requests`** (it lets Python visit websites).

In the **terminal** (not in the notebook), inside the class folder:

```bash
uv add requests
```

`uv` updates your project's shopping list (`pyproject.toml`) and installs it. If the next cell says `ModuleNotFoundError`, restart the kernel (menu: Kernel → Restart) and run again.

In [ ]:
import requests
print("requests version:", requests.__version__)

## 2️⃣ What is an API?

**API** = **A**pplication **P**rogramming **I**nterface.

**Analogy:** A **restaurant waiter**. 🍽️
- You (your program) don't walk into the kitchen.
- You give the **waiter** (the API) your **order** (a request).
- The kitchen (another computer, called a **server**) prepares it.
- The waiter brings back your **food** (the response).

```
Your Python code ──request──►  API (waiter)  ──►  Server (kitchen)
Your Python code ◄──response──  API (waiter)  ◄──  Server (kitchen)
```

Weather apps, maps, payment apps (like Paystack), and **AI chatbots** all work like this.

## 3️⃣ JSON — the language computers use to share data

Servers reply in **JSON**. Good news: **it looks just like Python dictionaries and lists!** You already know it.

In [ ]:
# This is a pretend API response (no internet needed)
response = {
    "city": "Awka",
    "current": {"temperature": 29.5, "condition": "Partly cloudy"},
    "next_days": [31, 30, 28],
}

print(response["city"])
print(response["current"]["temperature"])   # a dictionary inside a dictionary
print(response["next_days"][0])             # a list inside a dictionary

### ✏️ Quick practice (offline)
1. Print the `condition`.
2. Print the highest of the `next_days` temperatures.
3. Add a key called `"humidity"` to `response["current"]`.

In [ ]:
# Your turn

## 4️⃣ Your first real API call 🌐

We'll use **Open-Meteo**, a free weather API (no sign-up needed).

A **request** has an address (URL) and options (**parameters**). Awka is at latitude `6.21`, longitude `7.07`.

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 6.21, "longitude": 7.07, "current_weather": True}

response = requests.get(url, params=params, timeout=10)
print("Status code:", response.status_code)    # 200 means OK
data = response.json()                          # JSON → Python dictionary!
print(data)

### Status codes (the waiter's answers)

| Code | Meaning |
| ---- | ------- |
| **200** | ✅ OK, here's your food |
| **404** | 🤷 Not found (wrong address) |
| **401 / 403** | 🚫 Not allowed (need a key / permission) |
| **500** | 🔥 The kitchen is on fire (server error) |

In [ ]:
now = data["current_weather"]
print(f"Temperature in Awka: {now['temperature']}°C")
print(f"Wind speed: {now['windspeed']} km/h")

## 5️⃣ Wrap it in a function + Gradio 🌦️

Always use `try / except`. The internet can fail (no data, server down) and your app must not crash.

In [ ]:
CITIES = {
    "Awka": (6.21, 7.07),
    "Lagos": (6.52, 3.38),
    "Abuja": (9.08, 7.40),
    "Enugu": (6.46, 7.55),
    "Port Harcourt": (4.82, 7.05),
}   # 🎛️ Tutor: add the students' hometowns live!

def get_weather(city):
    lat, lon = CITIES[city]
    try:
        r = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current_weather": True},
            timeout=10,
        )
        r.raise_for_status()                 # raises an error for 404, 500, ...
        w = r.json()["current_weather"]
        temp = w["temperature"]
        if temp >= 32:
            advice = "🔥 Very hot. Drink water!"
        elif temp >= 25:
            advice = "☀️ Warm. Light clothes."
        else:
            advice = "🌧️ Cool. Maybe take a jacket."
        return f"{city}: {temp}°C, wind {w['windspeed']} km/h. {advice}"
    except requests.RequestException:
        return "😢 Couldn't reach the weather service. Check your internet and try again."

demo = gr.Interface(
    fn=get_weather,
    inputs=gr.Dropdown(list(CITIES), value="Awka", label="City"),
    outputs=gr.Textbox(label="Weather now"),
    title="🌦️ Live Weather App",
    flagging_mode="never",
)
show(demo)

## 6️⃣ APIs that ask for a key 🔑

Some APIs (including **AI APIs**) charge money or protect their data, so they give you a secret **API key**, like a **password for your program**.

🔒 **Golden rules of API keys**
1. **Never** type a key directly into code you share, post online, or push to GitHub.
2. Store it in an **environment variable** (a secret setting on your computer).
3. If a key leaks, delete it and make a new one right away.
4. A key is tied to **money** and **your identity**. Treat it like your bank card.

In [ ]:
import os
# Reading a secret from the environment (safe way). Returns None if not set.
key = os.getenv("MY_SECRET_KEY")
print("Key found?", key is not None)

## 🔨 Build along (with your tutor)
We build a **"Daily Dashboard"**: today's date, a weather line for the chosen city, and a random fun fact. We'll call a second API: `https://catfact.ninja/fact` (returns JSON with a `"fact"` key).

In [ ]:
# BUILD ALONG 1 - get a fact from the API

In [ ]:
# BUILD ALONG 2 - combine date + weather + fact into one Gradio app

## 🎯 Challenges

### ⭐ Starter (age 12)
Add 3 more cities to the weather app (find latitude/longitude by searching "<city> coordinates"). Add an emoji to the advice for each temperature range of your own choosing.

### 🚀 Level up (age 15)
Create a **3-day forecast** app. Add `"daily": "temperature_2m_max,temperature_2m_min"` and `"timezone": "Africa/Lagos"` to `params` (remove `current_weather`). The answer is in `data["daily"]`, which holds lists named `time`, `temperature_2m_max` and `temperature_2m_min`. Loop with `zip(...)` to print one line per day.

## 🤖 The AI boom connection

**This is it.** Talking to an AI model from Python is *exactly* what you did today:
1. Build a request (a dictionary of options, including your **API key** and the **messages**)
2. Send it with an API call
3. Read the JSON reply, then show it in Gradio

Next week you do that for real. 🧠

## ✅ Recap quiz
1. What is an API? Give the restaurant analogy. <details><summary>Answer</summary>A way for programs to talk to each other; the waiter takes your order to the kitchen and brings back the food.</details>
2. What is JSON like in Python? <details><summary>Answer</summary>Dictionaries and lists.</details>
3. What does status code 200 mean? <details><summary>Answer</summary>OK, it worked.</details>
4. Why must internet code use try/except? <details><summary>Answer</summary>Networks and servers can fail; the app should handle it gracefully.</details>
5. Where should an API key be stored? <details><summary>Answer</summary>In an environment variable, never in shared code.</details>

## 🏠 Homework
Improve the weather app: show the weather for **two** cities side by side (two Textboxes), and add a friendly error if the user's internet is off. Bring a screenshot to Week 6.

## 🔜 Next week
**🤖 AI in Python!** How AI actually works, then build your own **AI study-buddy chatbot**.